In [ ]:
import numpy as np
import matplotlib
matplotlib.use('TkAgg')  # Use TkAgg backend for interactive plotting
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from quat_utils import quatconj, quatmultiply
from h5_utils import save, load

In [ ]:
REF_DATA      = load('4_REFERENCED_DATA.h5', 'REF_DATA')
PING360_VALID = load('3_VALID_DATA.h5', 'PROFILO_PING360_VALID')
SEAKING_VALID = load('3_VALID_DATA.h5', 'PROFILO_SEAKING_VALID')

<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 20px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h3 style="color: #5d4e37; text-align: center; margin-bottom: 20px; font-weight: bold; max-width: 100%; word-wrap: break-word; hyphens: auto;">Navigation & Path Reconstruction Engine</h3>

<p style="margin: 0 0 15px 0; color: #4a4a4a; font-size: 0.9em;">
This process acts as the robot's master timeline, combining data from multiple sensors to map its continuous 3D journey. First, it finds the common time window where all sensors were recording. Then, it steps through time: whenever the robot reports a new speed or tilt, the engine translates those local movements into real-world coordinates and steps the position forward along the map, using the depth sensor to lock in the vertical position.
</p>

<div style="background-color: #f4eee1; padding: 15px; border-left: 4px solid #bda881; border-radius: 4px;">
<h4 style="margin-top: 0; color: #5d4e37; font-size: 0.95em; font-weight: bold;">Sonar Synchronization</h4>
<p style="margin: 0; color: #4a4a4a; font-size: 0.9em;">
The final part of the engine acts as a digital bookmarking system. Imagine the robot is continuously moving along its calculated path while periodically snapping "photos" of its surroundings using its two sonars (Ping360 and SeaKing). Because the sonars fire at their own pace, these specific lines of code constantly watch the master clock. The exact moment the clock catches up to the time a sonar ping was fired, the system writes down the robot's current position number on the path. This crucial step guarantees that later on, you know exactly where the robot was physically located when it captured every single sonar image.
</p>
</div>

</div>

In [ ]:
# Determine the common time interval
t_init = max([
    REF_DATA.QUATERNION_ECHO_TIME_STAMP[0],
    REF_DATA.QUATERNION_IMU_TIME_STAMP[0],
    REF_DATA.VEL_B_TIME_STAMP[0],
    REF_DATA.DEPTH_0_TIME_STAMP[0],
    PING360_VALID.ABS_TIME[0],
    SEAKING_VALID.ABS_TIME[0]
])
print(f"t_init = {t_init}")

t_end = min([
    REF_DATA.QUATERNION_ECHO_TIME_STAMP[-1],
    REF_DATA.QUATERNION_IMU_TIME_STAMP[-1],
    REF_DATA.VEL_B_TIME_STAMP[-1],
    REF_DATA.DEPTH_0_TIME_STAMP[-1],
    PING360_VALID.ABS_TIME[-1],
    SEAKING_VALID.ABS_TIME[-1]
])
print(f"t_end  = {t_end}")

count_IMU      = 0
count_VEL_B    = 0
count_DEPTH    = 0
count_X        = 0
count_PING360  = 0
count_SEAKING  = 0

X_ECHO = np.array([0.0, 0.0, 0.0])
X_IMU  = np.array([0.0, 0.0, 0.0])

CURRENT_IMU_QUAT  = [*REF_DATA.QUATERNION_IMU[0]]
CURRENT_ECHO_QUAT = [*REF_DATA.QUATERNION_ECHO[0]]
CURRENT_VEL_B     = np.array([0.0, 0.0, 0.0])

new_data = False
t_prev   = t_init

# Creating the containers
X_IMU_STOCK      = []
TIME_IMU_STOCK   = []
QUAT_IMU_STOCK   = []
V_0_IMU_STOCK    = []
PING360_POSE_IDX = []
SEAKING_POSE_IDX = []

for t in np.arange(t_init, t_end + 1):

    # Check for new IMU quaternion
    if REF_DATA.QUATERNION_IMU_TIME_STAMP[count_IMU] <= t:
        CURRENT_IMU_QUAT = REF_DATA.QUATERNION_IMU[count_IMU, :]
        count_IMU += 1
        new_data = True

    # Check for new body-frame velocity
    if REF_DATA.VEL_B_TIME_STAMP[count_VEL_B] <= t:
        CURRENT_VEL_B = REF_DATA.VEL_B[:, count_VEL_B]   # REF_DATA.VEL_B is shape (3, N)
        count_VEL_B += 1
        new_data = True

    # Check for new depth
    if REF_DATA.DEPTH_0_TIME_STAMP[count_DEPTH] <= t:
        CURRENT_DEPTH = REF_DATA.DEPTH_0[count_DEPTH]
        count_DEPTH += 1
        new_data = True

    if new_data:
        # Rotate the velocity by the IMU quaternion
        v_quat   = [0, *CURRENT_VEL_B]
        Q_TEMP   = quatmultiply(CURRENT_IMU_QUAT, quatmultiply(v_quat, quatconj(CURRENT_IMU_QUAT)))
        V_0_IMU  = np.array([Q_TEMP[1], Q_TEMP[2], Q_TEMP[3]])
        X_IMU   += V_0_IMU * (t - t_prev) / 1000
        X_IMU[2] = -CURRENT_DEPTH

        X_IMU_STOCK.append(X_IMU.copy())
        TIME_IMU_STOCK.append(t)
        QUAT_IMU_STOCK.append(CURRENT_IMU_QUAT.copy())
        V_0_IMU_STOCK.append(V_0_IMU.copy())

        new_data = False
        t_prev   = t
        count_X += 1

    while count_PING360 <= len(PING360_VALID.ABS_TIME) and \
            PING360_VALID.ABS_TIME[count_PING360] <= t:
        PING360_POSE_IDX.append(count_X)
        count_PING360 += 1

    while count_SEAKING <= len(SEAKING_VALID.ABS_TIME) and \
            SEAKING_VALID.ABS_TIME[count_SEAKING] <= t:
        SEAKING_POSE_IDX.append(count_X)
        count_SEAKING += 1

# Build trajectory arrays (kept in memory — no intermediate save)
X_IMU_ARRAY    = np.array(X_IMU_STOCK)     # shape (N, 3)
QUAT_IMU_ARRAY = np.array(QUAT_IMU_STOCK)  # shape (N, 4)
H              = np.hstack((X_IMU_ARRAY, QUAT_IMU_ARRAY))  # shape (N, 7)
TIME           = np.array(TIME_IMU_STOCK)

print(f"Trajectory computed: {H.shape[0]} poses")
print(f"PING360_POSE_IDX: {len(PING360_POSE_IDX)} entries")
print(f"SEAKING_POSE_IDX: {len(SEAKING_POSE_IDX)} entries")

<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 20px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h2 style="color: #5d4e37; text-align: center; margin-bottom: 20px; font-weight: bold; max-width: 100%; word-wrap: break-word; hyphens: auto;">Sonar Data Treatment and Impact Detection</h2>

<p style="text-align: justify; margin: 20px 0; color: #4a4a4a; width: 100%; max-width: 100%; word-wrap: break-word; overflow-wrap: break-word; hyphens: auto; box-sizing: border-box;">
This section performs <strong>sonar data processing and object detection</strong> from the NavScoot 2 recordings. The process extracts meaningful environmental information from sonar intensity data by identifying the first strong echoes in each ping, which represent obstacles or seafloor features. The algorithm processes <strong>profiling sonar skylines</strong> to detect impacts and organizes them for 3D mapping applications.
</p>

<div style="overflow-x: auto; margin: 20px 0; max-width: 100%;">
<table style="width: 100%; min-width: 600px; border-collapse: collapse; font-size: clamp(0.7em, 2vw, 0.9em); background-color: white; border-radius: 6px; overflow: hidden; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
  <thead>
    <tr style="background-color: #8b7355;">
      <th style="padding: clamp(8px, 2vw, 12px); text-align: left; font-weight: bold; border-right: 1px solid #a0937a; color: white; word-wrap: break-word;">Step</th>
      <th style="padding: clamp(8px, 2vw, 12px); text-align: left; font-weight: bold; border-right: 1px solid #a0937a; color: white; word-wrap: break-word;">Process</th>
      <th style="padding: clamp(8px, 2vw, 12px); text-align: left; font-weight: bold; color: white; word-wrap: break-word;">Output</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #f9f7f1;">
      <td style="padding: clamp(6px, 1.5vw, 10px); font-weight: bold; border-right: 1px solid #e6e0d4; border-bottom: 1px solid #e6e0d4; color: #4a4a4a; word-wrap: break-word;">1</td>
      <td style="padding: clamp(6px, 1.5vw, 10px); border-right: 1px solid #e6e0d4; border-bottom: 1px solid #e6e0d4; color: #4a4a4a; word-wrap: break-word;">Interactive offset selection</td>
      <td style="padding: clamp(6px, 1.5vw, 10px); border-bottom: 1px solid #e6e0d4; color: #4a4a4a; word-wrap: break-word;">Cleaned skyline data (noise removal)</td>
    </tr>
    <tr style="background-color: white;">
      <td style="padding: clamp(6px, 1.5vw, 10px); font-weight: bold; border-right: 1px solid #e6e0d4; border-bottom: 1px solid #e6e0d4; color: #4a4a4a; word-wrap: break-word;">2</td>
      <td style="padding: clamp(6px, 1.5vw, 10px); border-right: 1px solid #e6e0d4; border-bottom: 1px solid #e6e0d4; color: #4a4a4a; word-wrap: break-word;">First strong return detection (Vertical)</td>
      <td style="padding: clamp(6px, 1.5vw, 10px); border-bottom: 1px solid #e6e0d4; color: #4a4a4a; word-wrap: break-word;">M_v: impact distances, angles, timestamps</td>
    </tr>
    <tr style="background-color: #f9f7f1;">
      <td style="padding: clamp(6px, 1.5vw, 10px); font-weight: bold; border-right: 1px solid #e6e0d4; border-bottom: 1px solid #e6e0d4; color: #4a4a4a; word-wrap: break-word;">3</td>
      <td style="padding: clamp(6px, 1.5vw, 10px); border-right: 1px solid #e6e0d4; border-bottom: 1px solid #e6e0d4; color: #4a4a4a; word-wrap: break-word;">First strong return detection (Horizontal)</td>
      <td style="padding: clamp(6px, 1.5vw, 10px); border-bottom: 1px solid #e6e0d4; color: #4a4a4a; word-wrap: break-word;">M_h: 3×n matrices, pose mappings</td>
    </tr>
    <tr style="background-color: white;">
      <td style="padding: clamp(6px, 1.5vw, 10px); font-weight: bold; border-right: 1px solid #e6e0d4; color: #4a4a4a; word-wrap: break-word;">4</td>
      <td style="padding: clamp(6px, 1.5vw, 10px); border-right: 1px solid #e6e0d4; color: #4a4a4a; word-wrap: break-word;">Data export</td>
      <td style="padding: clamp(6px, 1.5vw, 10px); color: #4a4a4a; word-wrap: break-word;">Single .h5 with H, TIME, M_v, M_h, PING360_POSE_IDX, SEAKING_POSE_IDX</td>
    </tr>
  </tbody>
</table>
</div>

</div>

<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 20px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h3 style="color: #5d4e37; text-align: center; margin-bottom: 20px; font-weight: bold; max-width: 100%; word-wrap: break-word; hyphens: auto;">Detecting Unusable Areas at the Start of Skyline</h3>

<p style="text-align: justify; margin: 20px 0; color: #4a4a4a; width: 100%; max-width: 100%; word-wrap: break-word; overflow-wrap: break-word; hyphens: auto; box-sizing: border-box;">
This interactive step identifies and removes <strong>acoustic artifacts</strong> and near-field noise that appear at the beginning of sonar skylines. The algorithm displays each profile sequentially, allowing the user to manually click on the transition point where valid sonar data begins.
</p>

<div style="background-color: #fff9e6; padding: 15px; border-left: 4px solid #f39c12; margin: 20px 0; width: 100%; max-width: 100%; box-sizing: border-box; overflow: hidden;">
<p style="margin: 0; color: #4a4a4a; word-wrap: break-word; overflow-wrap: break-word;"><strong>PS:</strong> The calculations should complete very quickly, usually finishing immediately after clicking the X button to close the interactive window. If the process takes longer than <strong>10 seconds</strong>, press the <span style="color: #d73027; font-weight: bold;">interrupt button</span>, then restart the kernel and run the script again from the beginning.</p>
</div>

</div>

In [ ]:
# Load profiling sonar data (already loaded above; use the same objects)
PROFILO_SEAKING_VALID = SEAKING_VALID
PROFILO_PING360_VALID = PING360_VALID

def run_interactive_offset_selection():
    offset_x = []
    plt.figure(figsize=(12, 6))
    total_profiles = len(PROFILO_SEAKING_VALID.ABS_TIME)

    for i in range(total_profiles):
        plt.clf()
        nbr_bins = int(PROFILO_SEAKING_VALID.NBR_BINS[i])
        plt.plot(PROFILO_SEAKING_VALID.SKYLINE[i, 0:nbr_bins])
        plt.title(f'Profile {i+1}/{total_profiles}')
        plt.figtext(0.6, 0.7, 'Detection of undevelopable areas at the beginning of the Skyline', ha='center', fontsize=12)
        plt.figtext(0.6, 0.65, 'Click on the graph to define the threshold', ha='center', fontsize=12)
        plt.figtext(0.6, 0.6, 'Click X(Close) to finish', ha='center', fontsize=12)
        plt.xlabel('Bin', labelpad=10)
        plt.ylabel('Intensity of the Echo that the sonar captured', labelpad=20, fontsize=15)
        plt.grid(True, alpha=0.4)
        plt.draw()
        response = plt.ginput(1)
        if not response or response[0][0] <= 0:
            print("Finishing offset selection.")
            break
        offset_x.append(response[0][0])
        print(f"Selected offset for profile {i+1}: {response[0][0]}")
    plt.close()
    return offset_x

if __name__ == "__main__":
    offset_x = run_interactive_offset_selection()
    if offset_x:
        offset_bins = 1 + int(np.floor(max(offset_x)))
        print(f"Final offset_bins: {offset_bins}")
        PROFILO_SEAKING_VALID.SKYLINE[:, 0:offset_bins] = 0

In [ ]:


def run_interactive_offset_selection():
    offset_x = []
    plt.figure(figsize=(12, 6))
    total_profiles = len(PROFILO_PING360_VALID.ABS_TIME)

    for i in range(total_profiles):
        plt.clf()
        nbr_bins = int(PROFILO_PING360_VALID.NBR_BINS[i])
        plt.plot(PROFILO_PING360_VALID.SKYLINE[i, 0:nbr_bins])
        plt.title(f'Profile {i+1}/{total_profiles}')
        plt.figtext(0.6, 0.7, 'Detection of undevelopable areas at the beginning of the Skyline', ha='center', fontsize=12)
        plt.figtext(0.6, 0.65, 'Click on the graph to define the threshold', ha='center', fontsize=12)
        plt.figtext(0.6, 0.6, 'Click X(Close) to finish', ha='center', fontsize=12)
        plt.xlabel('Bin', labelpad=10)
        plt.ylabel('Intensity of the Echo that the sonar captured', labelpad=20, fontsize=15)
        plt.grid(True, alpha=0.4)
        plt.draw()
        response = plt.ginput(1)
        if not response or response[0][0] <= 0:
            print("Finishing offset selection.")
            break
        offset_x.append(response[0][0])
        print(f"Selected offset for profile {i+1}: {response[0][0]}")
    plt.close()
    return offset_x

if __name__ == "__main__":
    offset_x = run_interactive_offset_selection()
    if offset_x:
        offset_bins = 1 + int(np.floor(max(offset_x)))
        print(f"Final offset_bins: {offset_bins}")
        PROFILO_PING360_VALID.SKYLINE[:, 0:offset_bins] = 0

# Vertical Sonar — First Strong Return Detection

In [ ]:
threshold = 2   # Minimum echo intensity to consider an actual detected object

Vertical_DIST        = []
Vertical_ANGLE       = []
Vertical_GLOBAL_TIME = []
M_v                  = []

for i in range(len(SEAKING_POSE_IDX)):
    Length_Skyline = int(PROFILO_SEAKING_VALID.NBR_BINS[i])
    def_bin        = PROFILO_SEAKING_VALID.RANGE[i] / PROFILO_SEAKING_VALID.NBR_BINS[i]
    skyline_slice  = PROFILO_SEAKING_VALID.SKYLINE[i, 0:Length_Skyline]
    Impact_Indices = np.where(skyline_slice > threshold)[0]

    if len(Impact_Indices) > 0:
        DISTANCE = def_bin * Impact_Indices[0]

        if DISTANCE < PROFILO_SEAKING_VALID.RANGE[i]:
            m_v = {
                'rho': DISTANCE,
                'psi': PROFILO_SEAKING_VALID.ANGLE[i],
                'pose_idx': SEAKING_POSE_IDX[i],
                'time': PROFILO_SEAKING_VALID.ABS_TIME[i]
            }
            M_v.append(m_v)

# Horizontal Sonar — First Strong Return Detection

In [ ]:
threshold = 0.5   # Minimum echo intensity to consider an actual detected object

Horizontal_DIST        = []
Horizontal_ANGLE       = []
Horizontal_GLOBAL_TIME = []
M_h                    = []

for i in range(len(PING360_POSE_IDX)):
    Length_Skyline = int(PROFILO_PING360_VALID.NBR_BINS[i])
    def_bin        = 50 / PROFILO_PING360_VALID.NBR_BINS[i]
    skyline_slice  = PROFILO_PING360_VALID.SKYLINE[i, 0:Length_Skyline]
    Impact_Indices = np.where(skyline_slice > threshold)[0]

    if len(Impact_Indices) > 0:
        distances       = def_bin * Impact_Indices
        valid_distances = distances[distances < 50]

        n = len(valid_distances)
        if n > 0:
            row_rho   = valid_distances
            row_theta = np.full(n, np.nan)
            row_psi   = np.full(n, PROFILO_PING360_VALID.ANGLE[i])
            M_i_h     = np.vstack((row_rho, row_theta, row_psi))

            beam_data = {
                'M_i_h': M_i_h,
                'pose_idx': PING360_POSE_IDX[i],
                'time': PROFILO_PING360_VALID.ABS_TIME[i]
            }
            M_h.append(beam_data)

# Save All Results to a Single HDF5 File

In [ ]:
# --- Trajectory data ---
traj_dict = {
    'H'    : H,
    'TIME' : TIME
}

# --- Pose index arrays ---
pose_dict = {
    'PING360_POSE_IDX' : np.array(PING360_POSE_IDX, dtype=np.int64),
    'SEAKING_POSE_IDX' : np.array(SEAKING_POSE_IDX, dtype=np.int64)
}

# --- Vertical sonar (M_v) ---
M_v_dict = {
    'rho'      : np.array([m['rho']      for m in M_v], dtype=np.float64),
    'psi'      : np.array([m['psi']      for m in M_v], dtype=np.float64),
    'pose_idx' : np.array([m['pose_idx'] for m in M_v], dtype=np.float64),
    'time'     : np.array([m['time']     for m in M_v], dtype=np.float64)
}

# --- Horizontal sonar (M_h) ---
if len(M_h) > 0:
    M_h_matrix   = np.hstack([m['M_i_h'] for m in M_h]).astype(np.float64)
    M_h_pose_idx = np.concatenate([np.full(m['M_i_h'].shape[1], m['pose_idx']) for m in M_h]).astype(np.float64)
    M_h_time     = np.concatenate([np.full(m['M_i_h'].shape[1], m['time'])     for m in M_h]).astype(np.float64)
else:
    M_h_matrix   = np.array([], dtype=np.float64)
    M_h_pose_idx = np.array([], dtype=np.float64)
    M_h_time     = np.array([], dtype=np.float64)

M_h_dict = {
    'M'        : M_h_matrix,
    'pose_idx' : M_h_pose_idx,
    'time'     : M_h_time
}

# --- Write everything to one file ---
OUTPUT_FILE = 'HDF5_DATA/6_TRAJECTORY_AND_SONAR.h5'

save(OUTPUT_FILE, 'TRAJ',             traj_dict, mode='w')  # 'w' creates / overwrites the file
save(OUTPUT_FILE, 'POSE_IDX',         pose_dict)
save(OUTPUT_FILE, 'M_v',              M_v_dict)
save(OUTPUT_FILE, 'M_h',              M_h_dict)